# The four levels of LangChain abstraction

LangChain and LangGraph are best understood as a stack of four layers, where each layer is built on the one below. As you climb the stack, you hand more decisions to the framework in exchange for writing less code.

| Layer | Package | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state and routing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | planning, a filesystem and sub-agents | your intent |

In this notebook we take one small idea, a share price lookup, and rebuild it at each layer so you can feel the abstraction rising. The whole walk takes a few minutes.

## Before you start: your OpenAI API key

This notebook needs an OpenAI API key, stored as a Colab secret so it never appears in the notebook itself. Click the key icon in the left sidebar to open Secrets, choose "Add new secret", set the name to `OPENAI_API_KEY`, paste your key as the value, and switch on Notebook access. The cells below install the packages and read the key in.

Optionally, you can add a second secret called `ALPHAVANTAGE_API_KEY` (a free key from [alphavantage.co](https://www.alphavantage.co/support/#api-key), good for 25 requests a day) and the share price tool will fetch live quotes. Without it the notebook still works fully, using a small built-in price table.

In [ ]:
%pip install -qU langchain langchain-openai langgraph deepagents

In [ ]:
import os
import requests
from typing import Annotated
from typing_extensions import TypedDict
from IPython.display import Image, Markdown, display
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.agents import create_agent
from deepagents import create_deep_agent
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
try:
    os.environ["ALPHAVANTAGE_API_KEY"] = userdata.get("ALPHAVANTAGE_API_KEY")
except Exception:
    pass

if os.environ["OPENAI_API_KEY"] and os.environ["OPENAI_API_KEY"].startswith("sk-proj-"):
    print("OPENAI_API_KEY loaded, and it starts with sk-proj- as expected")
else:
    print("A key was loaded, but it does not start with sk-proj- so double check it is an OpenAI project key")

if os.environ.get("ALPHAVANTAGE_API_KEY"):
    print("ALPHAVANTAGE_API_KEY found, so share prices will be live")
else:
    print("No ALPHAVANTAGE_API_KEY set, so share prices will come from a built-in table")

## Layer 1: the building blocks

This is where LangChain began, as an abstraction layer over model providers. `ChatOpenAI` wraps the OpenAI API, and `invoke` is the one method to remember: you will see it again at every layer.

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")

reply = llm.invoke("In one sentence, what does it mean to say that an AI Agent is 'autonomous'?")
print(reply.content)

### Tools, and the loop you run yourself

The `@tool` decorator turns a Python function into something the model can call. Your docstring becomes the description the model reads, and your type hints become the argument schema.

Our tool looks up a share price. If you added the optional Alpha Vantage key it fetches a live quote, and if that fails for any reason, or there is no key, it quietly falls back to a small pretend price table so everything still runs.

At this layer the model can only ask for a tool. Watch what comes back when we bind the tool and pose a question: no answer, just a request in `tool_calls`.

In [ ]:
def fetch_live_price(symbol: str) -> float:
    response = requests.get("https://www.alphavantage.co/query", timeout=10, params={
        "function": "GLOBAL_QUOTE", "symbol": symbol, "apikey": os.environ["ALPHAVANTAGE_API_KEY"]})
    return float(response.json()["Global Quote"]["05. price"])

@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    if os.environ.get("ALPHAVANTAGE_API_KEY"):
        try:
            return fetch_live_price(symbol)
        except Exception:
            pass
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "GOOGL": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print(response.tool_calls)

Running the tool and handing the result back is our job at Layer 1. We run it, wrap the result in a `ToolMessage`, and invoke again so the model can finish. This little dance is the tool loop, written by hand, and everything above this layer exists to write it for us.

In [ ]:
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

for call in ai_message.tool_calls:
    result = get_share_price.invoke(call["args"])
    conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

print(llm_with_tools.invoke(conversation).content)

## Layer 2: LangGraph orchestration

Rather than running the loop by hand, we can describe it as a graph and let LangGraph run it. Nodes are just Python functions, a shared state flows between them, and edges decide what runs next. Two prebuilt pieces do the heavy lifting: `ToolNode` runs whatever tools the model asked for, and `tools_condition` routes to the tools node when the model wants a tool and to the end when it has answered.

The recipe is always the same: define the state, add nodes, add edges, compile. Then we draw it.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode([get_share_price]))
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "Which is more expensive, Apple stock or Google stock?"}]})
print(result["messages"][-1].content)

## Layer 3: the agent

`create_agent` gives you the whole loop from one function call. You hand it a model, tools and a prompt, and it builds the agent for you. Notice there is no graph code and no loop code left.

In [ ]:
agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[get_share_price],
    system_prompt="You are a helpful financial assistant. Use your tools.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "What are the share prices of Apple, Google and Amazon?"}]})
print(result["messages"][-1].content)

### The reveal

Here is the moment that ties the stack together. `create_agent` returns a compiled LangGraph graph, so we can draw it with exactly the same call we used a minute ago. Look at the shape: it is the graph we built by hand at Layer 2.

In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))

## Layer 4: the harness

A Deep Agent takes `create_agent` and wraps it in an opinionated harness for bigger, multi-step work. Three things come built in: a planning tool so the agent can write itself a todo list, a filesystem so it can save its work, and sub-agents it can delegate to. You supply the intent, and the harness supplies the structure.

We give it the same single tool as before, and a task with several steps. Watch the list of tool calls afterwards: it plans first, works through the lookups, then writes its answer to a file, all unprompted.

In [ ]:
deep_agent = create_deep_agent(
    model="openai:gpt-5.4-mini",
    tools=[get_share_price],
    system_prompt="You are an analyst. Plan your work with your todo tool, use your tools, and write your answer to a file when asked as a bulleted list.",
)

result = deep_agent.invoke({"messages": [{"role": "user", "content":
    "Look up the share prices of AAPL, GOOG and AMZN, then write a short markdown note ranking them to prices.md"}]})

tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the agent called, in order:")
print(tools_used)

The file lives in the agent's virtual filesystem, which travels in the result state. Here is the note it wrote.

In [ ]:
display(Markdown(result["files"]["/prices.md"]["content"].replace('$','')))

## The whole stack in one sitting

You have now run the same idea at four altitudes: a raw model call where you ran the tool loop yourself, a LangGraph graph that ran it for you, a one-line agent that turned out to be that same graph underneath, and a harness that planned, worked and wrote files on its own.

The four layers form a stack you tap at any altitude, not a ladder you climb and abandon. Real applications mix them freely, perhaps a `create_agent` for the main loop with a drop down to LangGraph for one custom flow. This notebook is a tiny taste of Week 4 of the AI Engineer Agentic Track, where each of these layers gets a full day.